# StreamView Analytics: preparación del catálogo
**Visualización de Datos · Informática · Duoc UC**  
**Audiencia:** Gerencia de Marketing.

Trabajamos con el archivo entregado, sin inventar ni completar cifras. Preparamos cuatro CSV para que el dashboard Dash + Plotly solo los lea. Usamos únicamente pandas y numpy en el código de análisis. Ejecutamos las celdas en orden en Colab, después de subir el archivo desde el panel **Archivos**. No instalamos paquetes ni conectamos fuentes externas.

Mantenemos la clave compuesta `(show_id, type)` en todas las operaciones. No creamos gráficos; reservamos `release_year` únicamente como filtro del dashboard.

## 1. Carga e inspección — CELDA DE TEXTO
Cargamos el CSV y revisamos su tamaño, columnas, tipos, nulos y cinco ejemplos antes de modificarlo. Guardamos una copia para comparar el antes y el después. Si no encontramos el archivo, lo subimos al directorio de trabajo de Colab.

In [ ]:
# CELDA DE CÓDIGO 1
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
ARCHIVO = 'catalogo_con_finanzas.csv'
CLAVE = ['show_id', 'type']
original = pd.read_csv(ARCHIVO)
print('Shape original:', original.shape)
display(pd.DataFrame({'tipo': original.dtypes.astype(str),
                      'nulos': original.isna().sum(),
                      'porcentaje_nulos': original.isna().mean().mul(100)}))
display(original.head(5))

Shape original: (31991, 16)


,tipo,nulos,porcentaje_nulos
show_id,int64,0,0.000000
type,object,0,0.000000
title,object,0,0.000000
director,object,11092,34.672252
cast,object,1360,4.251196
country,object,2261,7.067613
date_added,object,0,0.000000
release_year,int64,0,0.000000
genres,object,1079,3.372824
language,object,0,0.000000


,show_id,type,title,director,cast,country,date_added,release_year,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0


Shape original: (31991, 16)


,tipo,nulos,porcentaje_nulos
show_id,int64,0,0.000000
type,object,0,0.000000
title,object,0,0.000000
director,object,11092,34.672252
cast,object,1360,4.251196
country,object,2261,7.067613
date_added,object,0,0.000000
release_year,int64,0,0.000000
genres,object,1079,3.372824
language,object,0,0.000000


,show_id,type,title,director,cast,country,date_added,release_year,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0


## 2. Integración — CELDA DE TEXTO
Partimos de un archivo que ya integra películas y series. Según la procedencia informada, `budget` y `revenue` se unieron antes desde el archivo original de películas; no repetimos esa unión ni afirmamos haber auditado el archivo original, que no recibimos. Comprobamos la distribución por tipo y las finanzas vacías de las series.

No usamos `show_id` solo: un mismo número puede identificar una película y una serie distintas. Si aparecieran duplicados de la clave compuesta, detenemos el proceso para revisarlos en lugar de borrar registros.

In [ ]:
# CELDA DE CÓDIGO 2
REQUERIDAS = CLAVE + ['title', 'country', 'genres', 'language',
    'release_year', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
assert set(REQUERIDAS).issubset(original.columns), 'Faltan columnas requeridas'
assert not original[CLAVE].isna().any().any(), 'Hay claves nulas'
assert not original.duplicated(CLAVE).any(), 'Revisamos claves compuestas duplicadas'
conteo_original = original.groupby('type').size().rename('titulos')
display(conteo_original.to_frame())
ids_cruzados = int(original.groupby('show_id')['type'].nunique().gt(1).sum())
print('IDs presentes en ambos tipos:', ids_cruzados)
assert original.loc[original['type'].eq('TV Show'), ['budget', 'revenue']].isna().all().all()
print('Finanzas de series: todas ausentes, como corresponde a esta fuente.')

,titulos
type,
Movie,16000
TV Show,15991


IDs presentes en ambos tipos: 397
Finanzas de series: todas ausentes, como corresponde a esta fuente.


,titulos
type,
Movie,16000
TV Show,15991


IDs presentes en ambos tipos: 397
Finanzas de series: todas ausentes, como corresponde a esta fuente.


## 3. Limpieza con antes y después — CELDA DE TEXTO
Quitamos espacios externos, convertimos cadenas vacías a nulos y descartamos `rating` y `duration` si existen. No imputamos popularidad, votos ni finanzas. Convertimos ceros financieros en NaN; también tratamos valores negativos o no finitos como inválidos, dejando un conteo auditable.

Usamos “Sin país”, “Sin género” y “Sin idioma” solo como etiquetas de ausencia. No las consideramos categorías reales. Conservamos `Unknown` en el catálogo, pero lo excluimos de los análisis de géneros. Conservamos `xx` como código no informativo y lo excluimos de los KPIs/rankings de idiomas. No intentamos validar externamente todos los códigos. No eliminamos ninguna fila.

In [ ]:
# CELDA DE CÓDIGO 3
catalogo = original.copy()
catalogo = catalogo.drop(columns=['rating', 'duration'], errors='ignore')
for col in catalogo.select_dtypes(include=['object']).columns:
    catalogo[col] = catalogo[col].astype('string').str.strip().replace('', pd.NA)

numericas = ['release_year', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
conversiones = []
for col in numericas:
    previo = catalogo[col].notna()
    catalogo[col] = pd.to_numeric(catalogo[col], errors='coerce')
    conversiones.append({'columna': col,
                         'no_numericos': int((previo & catalogo[col].isna()).sum())})
display(pd.DataFrame(conversiones))

cambios_finanzas = []
for col in ['budget', 'revenue']:
    s = catalogo[col]
    cambios_finanzas.append({'columna': col, 'ceros_a_NaN': int(s.eq(0).sum()),
        'negativos_o_no_finitos': int((s.notna() & (s.lt(0) | ~np.isfinite(s))).sum())})
    catalogo[col] = s.where(s.gt(0) & np.isfinite(s), np.nan)
display(pd.DataFrame(cambios_finanzas))

for col, etiqueta in [('country', 'Sin país'), ('genres', 'Sin género'),
                       ('language', 'Sin idioma')]:
    catalogo[col] = catalogo[col].fillna(etiqueta)

INVALIDOS_PAIS = {'sin país', 'unknown', 'nan', 'none', 'null', 'n/a'}
INVALIDOS_GENERO = {'sin género', 'unknown', 'nan', 'none', 'null', 'n/a'}
INVALIDOS_IDIOMA = {'sin idioma', 'unknown', 'xx', 'und', 'nan', 'none', 'null', 'n/a'}
idioma_real = ~catalogo['language'].str.casefold().isin(INVALIDOS_IDIOMA)

comparacion = pd.DataFrame({'tipo_antes': original.dtypes.astype(str),
    'nulos_antes': original.isna().sum(),
    'tipo_despues': catalogo.dtypes.astype(str),
    'nulos_despues': catalogo.isna().sum()})
display(comparacion)
display(pd.DataFrame({'etapa': ['Antes', 'Después'],
    'filas': [len(original), len(catalogo)],
    'columnas': [original.shape[1], catalogo.shape[1]],
    'duplicados_clave': [original.duplicated(CLAVE).sum(), catalogo.duplicated(CLAVE).sum()]}))
print('Filas eliminadas:', len(original) - len(catalogo))
assert not catalogo[CLAVE].isna().any().any()
assert not catalogo.duplicated(CLAVE).any()
# Los títulos sin votos no tienen valoración real: el 0 significa "sin dato"
catalogo['valoracion'] = catalogo['vote_average'].where(catalogo['vote_count'] > 0)
print('Títulos sin votos (valoración vacía):', int(catalogo['valoracion'].isna().sum()))

,columna,no_numericos
0,release_year,0
1,popularity,0
2,vote_count,0
3,vote_average,0
4,budget,0
5,revenue,0


,columna,ceros_a_NaN,negativos_o_no_finitos
0,budget,11153,0
1,revenue,10355,0


,tipo_antes,nulos_antes,tipo_despues,nulos_despues
show_id,int64,0,int64,0
type,object,0,string,0
title,object,0,string,0
director,object,11092,string,11092
cast,object,1360,string,1360
country,object,2261,string,0
date_added,object,0,string,0
release_year,int64,0,int64,0
genres,object,1079,string,0
language,object,0,string,0


,etapa,filas,columnas,duplicados_clave
0,Antes,31991,16,0
1,Después,31991,16,0


Filas eliminadas: 0
Títulos sin votos (valoración vacía): 4560


,columna,no_numericos
0,release_year,0
1,popularity,0
2,vote_count,0
3,vote_average,0
4,budget,0
5,revenue,0


,columna,ceros_a_NaN,negativos_o_no_finitos
0,budget,11153,0
1,revenue,10355,0


,tipo_antes,nulos_antes,tipo_despues,nulos_despues
show_id,int64,0,int64,0
type,object,0,string,0
title,object,0,string,0
director,object,11092,string,11092
cast,object,1360,string,1360
country,object,2261,string,0
date_added,object,0,string,0
release_year,int64,0,int64,0
genres,object,1079,string,0
language,object,0,string,0


,etapa,filas,columnas,duplicados_clave
0,Antes,31991,16,0
1,Después,31991,16,0


Filas eliminadas: 0
Títulos sin votos (valoración vacía): 4560


## 4. Finanzas: solo películas — CELDA DE TEXTO
Calculamos `retorno = revenue / budget` solo cuando ambos importes son positivos. Es un múltiplo de ingresos sobre presupuesto, no utilidad ni ROI: no incorpora marketing, distribución ni otros costos. No suponemos una moneda porque el CSV no la documenta.

Mostramos el número y porcentaje de películas con presupuesto, ingresos y ambos disponibles; el denominador siempre es el total de películas. Definimos presupuestos sospechosos como positivos menores a 1.000 o mayores a 1.000.000.000, en las unidades originales. Es una regla operativa de revisión, no una prueba de error. Los marcamos, no los borramos ni los reemplazamos; los KPIs incluyen todos los positivos y ofrecemos una comparación sin los sospechosos.

In [ ]:
# CELDA DE CÓDIGO 4
catalogo['retorno'] = np.nan
catalogo['budget_sospechoso'] = False
es_pelicula = catalogo['type'].eq('Movie')
ambos = es_pelicula & catalogo['budget'].gt(0) & catalogo['revenue'].gt(0)
catalogo.loc[ambos, 'retorno'] = (catalogo.loc[ambos, 'revenue'] /
                                      catalogo.loc[ambos, 'budget'])
BUDGET_MIN_REVISION = 1_000
BUDGET_MAX_REVISION = 1_000_000_000
sospechoso = es_pelicula & catalogo['budget'].gt(0) & (
    catalogo['budget'].lt(BUDGET_MIN_REVISION) |
    catalogo['budget'].gt(BUDGET_MAX_REVISION))
catalogo.loc[sospechoso, 'budget_sospechoso'] = True
peliculas_finanzas = catalogo.loc[es_pelicula].copy()
n_peliculas = len(peliculas_finanzas)
cobertura = pd.DataFrame({'indicador': ['Presupuesto', 'Ingresos', 'Ambos / retorno'],
    'n': [peliculas_finanzas['budget'].notna().sum(),
          peliculas_finanzas['revenue'].notna().sum(),
          peliculas_finanzas['retorno'].notna().sum()]})
cobertura['porcentaje_peliculas'] = cobertura['n'].div(n_peliculas).mul(100)
display(cobertura)
print('Presupuestos sospechosos:', int(sospechoso.sum()))
display(peliculas_finanzas.loc[peliculas_finanzas['budget_sospechoso'],
    CLAVE + ['title', 'budget', 'revenue', 'retorno']].sort_values('budget'))
display(pd.DataFrame({
    'todos_los_positivos': peliculas_finanzas[['budget', 'revenue', 'retorno']].median(),
    'sin_budget_sospechoso': peliculas_finanzas.loc[
        ~peliculas_finanzas['budget_sospechoso'], ['budget', 'revenue', 'retorno']].median()
}))

,indicador,n,porcentaje_peliculas
0,Presupuesto,4847,30.29375
1,Ingresos,5645,35.28125
2,Ambos / retorno,3540,22.12500


Presupuestos sospechosos: 55


,show_id,type,title,budget,revenue,retorno
12572,916405,Movie,The Quiet Girl,1.0,6.0,6.000000e+00
9132,571055,Movie,Down,1.0,NaN,NaN
1917,91551,Movie,Death of a Superhero,3.0,NaN,NaN
1309,66125,Movie,Red Dog,5.0,13.0,2.600000e+00
6269,391698,Movie,The Beatles: Eight Days a Week - The Touring Y...,5.0,12283800.0,2.456760e+06
13648,715385,Movie,No Dogs or Italians Allowed,5.0,1395922.0,2.791844e+05
2981,141581,Movie,Nous York,5.0,NaN,NaN
15827,1430181,Movie,The Walker,5.0,NaN,NaN
15851,1430188,Movie,Warp Weft Water Weeds,5.0,NaN,NaN
15847,1426985,Movie,The Oliver Theory,10.0,NaN,NaN


,todos_los_positivos,sin_budget_sospechoso
budget,1.050000e+07,11000000.0
revenue,8.555008e+06,8600192.0
retorno,1.700033e+00,1.7


,indicador,n,porcentaje_peliculas
0,Presupuesto,4847,30.29375
1,Ingresos,5645,35.28125
2,Ambos / retorno,3540,22.12500


Presupuestos sospechosos: 55


,show_id,type,title,budget,revenue,retorno
12572,916405,Movie,The Quiet Girl,1.0,6.0,6.000000e+00
9132,571055,Movie,Down,1.0,NaN,NaN
1917,91551,Movie,Death of a Superhero,3.0,NaN,NaN
1309,66125,Movie,Red Dog,5.0,13.0,2.600000e+00
6269,391698,Movie,The Beatles: Eight Days a Week - The Touring Y...,5.0,12283800.0,2.456760e+06
13648,715385,Movie,No Dogs or Italians Allowed,5.0,1395922.0,2.791844e+05
2981,141581,Movie,Nous York,5.0,NaN,NaN
15827,1430181,Movie,The Walker,5.0,NaN,NaN
15851,1430188,Movie,Warp Weft Water Weeds,5.0,NaN,NaN
15847,1426985,Movie,The Oliver Theory,10.0,NaN,NaN


,todos_los_positivos,sin_budget_sospechoso
budget,1.050000e+07,11000000.0
revenue,8.555008e+06,8600192.0
retorno,1.700033e+00,1.7


## 5. Explode separado — CELDA DE TEXTO
Separamos las listas de géneros y países en dos tablas independientes. Cada fila representa una asociación título–género o título–país. Nunca expandimos país y género juntos porque generaríamos un producto cartesiano y duplicaríamos conteos.

Conservamos las etiquetas de ausencia en los CSV para trazabilidad, pero generamos vistas válidas para KPIs y rankings. Eliminamos solamente asociaciones repetidas dentro de un mismo título usando `(show_id, type, categoría)`; no eliminamos títulos del catálogo. El total de asociaciones puede superar los 31.991 títulos.

In [ ]:
# CELDA DE CÓDIGO 5
BASE_EXPLODE = CLAVE + ['title', 'popularity', 'vote_average', 'valoracion', 'release_year']
def expandir(columna, nombre, etiqueta):
    tabla = catalogo[BASE_EXPLODE + [columna]].copy()
    tabla[columna] = tabla[columna].str.split(',')
    tabla = tabla.explode(columna).rename(columns={columna: nombre})
    tabla[nombre] = tabla[nombre].astype('string').str.strip().replace('', pd.NA).fillna(etiqueta)
    tabla = tabla.drop_duplicates(CLAVE + [nombre]).reset_index(drop=True)
    return tabla[CLAVE + ['title', nombre, 'popularity', 'vote_average', 'valoracion', 'release_year']]

generos = expandir('genres', 'género', 'Sin género')
paises = expandir('country', 'país', 'Sin país')
generos_validos = generos.loc[~generos['género'].str.casefold().isin(INVALIDOS_GENERO)].copy()
paises_validos = paises.loc[~paises['país'].str.casefold().isin(INVALIDOS_PAIS)].copy()
print('Asociaciones de géneros:', generos.shape)
print('Asociaciones de países:', paises.shape)
print('Asociaciones válidas:', len(generos_validos), 'de género y', len(paises_validos), 'de país')

Asociaciones de géneros: (66967, 8)
Asociaciones de países: (39889, 8)
Asociaciones válidas: 65789 de género y 37628 de país
Asociaciones de géneros: (66967, 8)
Asociaciones de países: (39889, 8)
Asociaciones válidas: 65789 de género y 37628 de país


## 6. KPIs con fórmula — CELDA DE TEXTO
Calculamos los KPIs generales desde el catálogo sin expandir para no sobreponderar títulos con varios países o géneros.

- **Total de títulos:** número de claves únicas `(show_id, type)`.
- **Porcentaje por tipo:** `100 × títulos del tipo / total de títulos`.
- **Popularidad:** mediana de `popularity`, nunca promedio.
- **Valoración:** media aritmética de `vote_average` solo en títulos con al menos un voto (`vote_count > 0`), guardada en la columna `valoracion`. Los títulos sin votos tienen vote_average 0 en la fuente, pero ese 0 significa ausencia de calificación, no mala calidad; por eso los dejamos vacíos. Conservamos `vote_average` original sin modificar.

- **Países reales:** número de etiquetas distintas de país válidas; respetamos los nombres de la fuente sin homologación geopolítica externa.
- **Idiomas reales:** número de códigos distintos informativos. No traducimos ni inventamos idiomas.
- **Idioma líder:** código con más títulos válidos; informamos su cantidad y porcentaje sobre títulos con idioma real. Desempatamos alfabéticamente.
- **Finanzas:** mediana de presupuestos positivos, mediana de ingresos positivos y mediana de los retornos individuales válidos de películas. La mediana de retornos no es el cociente de las otras dos medianas. Cada mediana omite sus NaN; mostramos su cobertura en la sección 4.

Las categorías “Sin país”, “Sin género” y “Unknown” no cuentan como países/géneros reales ni entran en rankings; los títulos que las contienen sí forman parte del total del catálogo.

In [ ]:
# CELDA DE CÓDIGO 6
n_total = len(catalogo)
conteo_tipo = catalogo.groupby('type').size()
idiomas_conteo = (catalogo.loc[idioma_real].groupby('language').size()
    .rename('cantidad').reset_index()
    .sort_values(['cantidad', 'language'], ascending=[False, True]))
lider = idiomas_conteo.iloc[0]
kpis = pd.DataFrame([
    ('Total títulos', n_total),
    ('Películas (%)', 100 * conteo_tipo['Movie'] / n_total),
    ('Series (%)', 100 * conteo_tipo['TV Show'] / n_total),
    ('Mediana popularidad', catalogo['popularity'].median()),
    ('Valoración media (títulos con votos)', catalogo['valoracion'].mean()),
    ('Títulos con votos', int(catalogo['valoracion'].notna().sum())),
    ('Países reales distintos', paises_validos['país'].nunique()),
    ('Idiomas reales distintos', catalogo.loc[idioma_real, 'language'].nunique()),
    ('Idioma líder (código)', lider['language']),
    ('Títulos idioma líder', int(lider['cantidad'])),
    ('Idioma líder (% con idioma real)', 100 * lider['cantidad'] / idioma_real.sum()),
    ('Mediana presupuesto películas', peliculas_finanzas['budget'].median()),
    ('Mediana ingresos películas', peliculas_finanzas['revenue'].median()),
    ('Mediana retorno películas (veces)', peliculas_finanzas['retorno'].median())
], columns=['indicador', 'valor'])
display(kpis)

,indicador,valor
0,Total títulos,31991
1,Películas (%),50.014066
2,Series (%),49.985934
3,Mediana popularidad,22.071
4,Valoración media (títulos con votos),6.630076
5,Títulos con votos,27431
6,Países reales distintos,147
7,Idiomas reales distintos,82
8,Idioma líder (código),en
9,Títulos idioma líder,13969


,indicador,valor
0,Total títulos,31991
1,Películas (%),50.014066
2,Series (%),49.985934
3,Mediana popularidad,22.071
4,Valoración media (títulos con votos),6.630076
5,Títulos con votos,27431
6,Países reales distintos,147
7,Idiomas reales distintos,82
8,Idioma líder (código),en
9,Títulos idioma líder,13969


## 7. Tablas para Marketing, sin gráficos — CELDA DE TEXTO
Armamos el Top 10 global con `vote_count >= 100` y orden descendente por popularidad. Para empates usamos cantidad de votos descendente y luego la clave compuesta ascendente.

Por género contamos títulos y calculamos su mediana de popularidad. Calculamos p75 sobre las medianas de los géneros válidos y p35 sobre sus cantidades; marcamos estrictamente `mediana > p75` y `cantidad < p35`, tanto por separado como en una señal conjunta de nicho. La señal no demuestra causalidad ni rentabilidad.

Comparamos series/películas y mostramos los 12 idiomas y países con más títulos, junto con su mediana de popularidad. Los porcentajes de idiomas usan títulos con idioma real; los de países usan títulos con al menos un país válido. Un título puede aparecer en varios países o géneros, por eso sus cantidades no se suman para obtener el catálogo. No hacemos análisis temporal.

In [ ]:
# CELDA DE CÓDIGO 7
TOP_COLUMNAS = CLAVE + ['title', 'popularity', 'vote_count', 'vote_average', 'release_year']
top10 = (catalogo.loc[catalogo['vote_count'].ge(100), TOP_COLUMNAS]
    .sort_values(['popularity', 'vote_count'] + CLAVE,
                 ascending=[False, False, True, True]).head(10))
print('Top 10:'); display(top10)

tabla_generos = (generos_validos.groupby('género').agg(
    cantidad=('show_id', 'size'), mediana_popularidad=('popularity', 'median')).reset_index())
p75 = tabla_generos['mediana_popularidad'].quantile(0.75)
p35 = tabla_generos['cantidad'].quantile(0.35)
tabla_generos['popularidad_mayor_p75'] = tabla_generos['mediana_popularidad'].gt(p75)
tabla_generos['cantidad_menor_p35'] = tabla_generos['cantidad'].lt(p35)
tabla_generos['nicho_destacado'] = (tabla_generos['popularidad_mayor_p75'] &
                                    tabla_generos['cantidad_menor_p35'])
tabla_generos = tabla_generos.sort_values(['cantidad', 'género'], ascending=[False, True])
print('Géneros. p75 popularidad:', p75, '| p35 cantidad:', p35); display(tabla_generos)

tabla_tipos = catalogo.groupby('type').agg(cantidad=('show_id', 'size'),
    mediana_popularidad=('popularity', 'median'),
    valoracion_media=('valoracion', 'mean'),
    titulos_con_votos=('valoracion', 'count'))
tabla_tipos['porcentaje_catalogo'] = tabla_tipos['cantidad'].div(n_total).mul(100)
print('Series vs películas:'); display(tabla_tipos)

tabla_idiomas = (catalogo.loc[idioma_real].groupby('language').agg(
    cantidad=('show_id', 'size'), mediana_popularidad=('popularity', 'median')).reset_index())
tabla_idiomas['porcentaje_con_idioma_real'] = tabla_idiomas['cantidad'].div(idioma_real.sum()).mul(100)
tabla_idiomas = tabla_idiomas.sort_values(['cantidad', 'language'], ascending=[False, True]).head(12)
print('Idiomas top 12:'); display(tabla_idiomas)

n_con_pais = len(paises_validos[CLAVE].drop_duplicates())
tabla_paises = paises_validos.groupby('país').agg(
    cantidad=('show_id', 'size'), mediana_popularidad=('popularity', 'median')).reset_index()
tabla_paises['porcentaje_con_pais_real'] = tabla_paises['cantidad'].div(n_con_pais).mul(100)
tabla_paises = tabla_paises.sort_values(['cantidad', 'país'], ascending=[False, True]).head(12)
print('Países top 12:'); display(tabla_paises)

Top 10:


,show_id,type,title,popularity,vote_count,vote_average,release_year
21000,63770,TV Show,The Late Show with Stephen Colbert,6421.923,288,6.493,2015
20000,59941,TV Show,The Tonight Show Starring Jimmy Fallon,4925.253,321,5.800,2014
15000,950396,Movie,The Gorge,3876.006,1572,7.785,2025
15001,1126166,Movie,Flight Risk,3320.616,350,6.000,2025
24000,81329,TV Show,Chronicles of the Sun,2987.485,117,6.800,2018
14000,762509,Movie,Mufasa: The Lion King,2643.627,1497,7.467,2024
14001,939243,Movie,Sonic the Hedgehog 3,2402.503,2047,7.742,2024
14002,1241982,Movie,Moana 2,2347.007,1758,7.100,2024
21001,62223,TV Show,The Late Late Show with James Corden,1953.221,132,5.400,2015
16000,33238,TV Show,Running Man,1929.898,187,8.241,2010


Géneros. p75 popularidad: 36.349000000000004 | p35 cantidad: 896.4999999999998


,género,cantidad,mediana_popularidad,popularidad_mayor_p75,cantidad_menor_p35,nicho_destacado
7,Drama,14769,22.6750,False,False,False
4,Comedy,9105,21.9150,False,False,False
3,Animation,4069,26.9370,False,False,False
23,Thriller,3769,12.3470,False,False,False
0,Action,3239,14.1040,False,False,False
5,Crime,3203,21.4880,False,False,False
8,Family,3014,26.1235,False,False,False
14,Mystery,2574,22.7980,False,False,False
17,Romance,2571,10.6300,False,False,False
11,Horror,2425,11.4070,False,False,False


Series vs películas:


...

,cantidad,mediana_popularidad,valoracion_media,titulos_con_votos,porcentaje_catalogo
type,,,,,
Movie,16000,10.9135,6.308877,15106,50.014066
TV Show,15991,36.2000,7.023750,12325,49.985934


Idiomas top 12:


,language,cantidad,mediana_popularidad,porcentaje_con_idioma_real
16,en,13969,17.0290,43.677694
80,zh,2834,29.4685,8.861234
34,ja,2784,24.2855,8.704897
40,ko,2240,25.7705,7.003940
17,es,1646,27.4395,5.146645
22,fr,1639,12.1590,5.124758
13,de,730,25.4615,2.282534
26,hi,635,24.5090,1.985492
73,tl,525,52.0530,1.641548
62,pt,483,35.9950,1.510225


Países top 12:


,país,cantidad,mediana_popularidad,porcentaje_con_pais_real
140,United States of America,10955,18.0330,36.848301
63,Japan,2869,24.3760,9.650185
138,United Kingdom,2608,16.7745,8.772284
27,China,2399,29.4190,8.069290
120,South Korea,2174,25.8150,7.312479
44,France,2108,11.2735,7.090481
23,Canada,1715,18.9310,5.768584
47,Germany,1160,16.7390,3.901783
56,India,912,18.3695,3.067608
121,Spain,822,15.2825,2.764884


## 8. Exportación — CELDA DE TEXTO
Exportamos cuatro CSV sin índice y con UTF-8. Guardamos los NaN como campos vacíos, no como ceros. Mostramos el shape y las cinco primeras filas de cada salida. El dashboard podrá leer estos archivos con `pd.read_csv` y recalcular filtros, tablas y KPIs sin consultar el archivo original.

El catálogo y las finanzas tienen una fila por clave compuesta; los archivos expandidos tienen una fila por clave compuesta y categoría. Si necesitamos filtrar por país y género a la vez, obtenemos claves distintas por separado y aplicamos su intersección al catálogo: no unimos las tablas expandidas directamente. Aplicamos las mismas exclusiones de categorías en el dashboard. Dejamos `release_year` como filtro, no como serie temporal.

In [ ]:
# CELDA DE CÓDIGO 8
salidas = {
    'catalogo_limpio.csv': catalogo,
    'generos.csv': generos,
    'paises.csv': paises,
    'peliculas_finanzas.csv': peliculas_finanzas
}
for nombre, tabla in salidas.items():
    tabla.to_csv(nombre, index=False, encoding='utf-8', na_rep='')
    print(nombre, '| shape:', tabla.shape)
    display(tabla.head(5))

catalogo_limpio.csv | shape: (31991, 19)


,show_id,type,title,director,cast,country,date_added,release_year,genres,language,description,popularity,vote_count,vote_average,budget,revenue,valoracion,retorno,budget_sospechoso
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0,6.380,4.561217,False
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0,8.369,5.243941,False
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0,7.744,3.817223,False
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0,7.600,2.278699,False
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0,7.800,2.999270,False


generos.csv | shape: (66967, 8)


,show_id,type,title,género,popularity,vote_average,valoracion,release_year
0,10192,Movie,Shrek Forever After,Comedy,203.893,6.38,6.38,2010
1,10192,Movie,Shrek Forever After,Adventure,203.893,6.38,6.38,2010
2,10192,Movie,Shrek Forever After,Fantasy,203.893,6.38,6.38,2010
3,10192,Movie,Shrek Forever After,Animation,203.893,6.38,6.38,2010
4,10192,Movie,Shrek Forever After,Family,203.893,6.38,6.38,2010


paises.csv | shape: (39889, 8)


...

,show_id,type,title,género,popularity,vote_average,valoracion,release_year
0,10192,Movie,Shrek Forever After,Comedy,203.893,6.38,6.38,2010
1,10192,Movie,Shrek Forever After,Adventure,203.893,6.38,6.38,2010
2,10192,Movie,Shrek Forever After,Fantasy,203.893,6.38,6.38,2010
3,10192,Movie,Shrek Forever After,Animation,203.893,6.38,6.38,2010
4,10192,Movie,Shrek Forever After,Family,203.893,6.38,6.38,2010


paises.csv | shape: (39889, 8)


,show_id,type,title,país,popularity,vote_average,valoracion,release_year
0,10192,Movie,Shrek Forever After,United States of America,203.893,6.380,6.380,2010
1,27205,Movie,Inception,United Kingdom,156.242,8.369,8.369,2010
2,27205,Movie,Inception,United States of America,156.242,8.369,8.369,2010
3,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,United Kingdom,121.191,7.744,7.744,2010
4,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,United States of America,121.191,7.744,7.744,2010


peliculas_finanzas.csv | shape: (16000, 19)


,show_id,type,title,director,cast,country,date_added,release_year,genres,language,description,popularity,vote_count,vote_average,budget,revenue,valoracion,retorno,budget_sospechoso
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0,6.380,4.561217,False
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0,8.369,5.243941,False
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0,7.744,3.817223,False
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0,7.600,2.278699,False
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0,7.800,2.999270,False


## 9. Validaciones — CELDA DE TEXTO
Comprobamos claves completas y únicas, valoraciones entre 0 y 10 y coherencia de los totales. Verificamos que cada título siga presente en las tablas expandidas, que no se dupliquen asociaciones y que las series no tengan finanzas imputadas. Validamos también los CSV después de leerlos, para no depender solo de los objetos en memoria. Si alguna regla falla, el assert detiene la ejecución; no lo ocultamos ni corregimos eliminando filas.

In [ ]:
# CELDA DE CÓDIGO 9
assert len(catalogo) == len(original) == 31_991
assert conteo_tipo.to_dict() == {'Movie': 16_000, 'TV Show': 15_991}
assert ids_cruzados == 397
assert not catalogo.duplicated(CLAVE).any()
assert not catalogo[CLAVE].isna().any().any()
assert catalogo['type'].isin(['Movie', 'TV Show']).all()
assert catalogo['vote_average'].notna().all()
assert catalogo['vote_average'].between(0, 10).all()
assert np.isfinite(catalogo['popularity']).all()
assert catalogo['vote_count'].ge(0).all()
assert conteo_tipo.sum() == n_total
assert np.isclose(tabla_tipos['porcentaje_catalogo'].sum(), 100)
assert not {'rating', 'duration'}.intersection(catalogo.columns)
assert catalogo.loc[~es_pelicula, ['budget', 'revenue', 'retorno']].isna().all().all()
assert catalogo[['budget', 'revenue']].gt(0).where(
    catalogo[['budget', 'revenue']].notna(), True).all().all()
assert np.array_equal(catalogo['retorno'].notna().to_numpy(), ambos.to_numpy(dtype=bool))
assert np.allclose(catalogo.loc[ambos, 'retorno'],
    catalogo.loc[ambos, 'revenue'] / catalogo.loc[ambos, 'budget'])


assert catalogo['valoracion'].dropna().between(0, 10).all()
assert np.array_equal(catalogo['valoracion'].isna().to_numpy(), catalogo['vote_count'].eq(0).to_numpy())
claves_catalogo = pd.MultiIndex.from_frame(catalogo[CLAVE])
for tabla, categoria in [(generos, 'género'), (paises, 'país')]:
    assert not tabla[CLAVE + [categoria]].isna().any().any()
    assert not tabla.duplicated(CLAVE + [categoria]).any()
    claves_tabla = pd.MultiIndex.from_frame(tabla[CLAVE].drop_duplicates())
    assert claves_catalogo.difference(claves_tabla).empty
    assert claves_tabla.difference(claves_catalogo).empty
assert tabla_generos['cantidad'].sum() == len(generos_validos)
assert idiomas_conteo['cantidad'].sum() == int(idioma_real.sum())
assert top10['vote_count'].ge(100).all() and len(top10) == 10
assert top10['popularity'].is_monotonic_decreasing
assert not generos_validos['género'].str.casefold().isin(INVALIDOS_GENERO).any()
assert not paises_validos['país'].str.casefold().isin(INVALIDOS_PAIS).any()

for nombre, tabla in salidas.items():
    leida = pd.read_csv(nombre)
    assert leida.shape == tabla.shape
    assert not leida[CLAVE].isna().any().any()
    if nombre in ['catalogo_limpio.csv', 'peliculas_finanzas.csv']:
        assert not leida.duplicated(CLAVE).any()
    else:
        categoria = 'género' if nombre == 'generos.csv' else 'país'
        assert not leida.duplicated(CLAVE + [categoria]).any()
print('Validaciones aprobadas. Conservamos 31.991 títulos y eliminamos 0 filas por ID.')

Validaciones aprobadas. Conservamos 31.991 títulos y eliminamos 0 filas por ID.
Validaciones aprobadas. Conservamos 31.991 títulos y eliminamos 0 filas por ID.


## Cierre: resumen de cinco líneas — CELDA DE TEXTO
Resumimos los resultados calculados sobre este archivo; las cinco líneas siguientes se generan a partir de los datos y no de cifras escritas manualmente.

In [ ]:
# CELDA DE CÓDIGO 10
print(f'1. Conservamos {n_total:,} títulos: {conteo_tipo["Movie"]:,} películas y {conteo_tipo["TV Show"]:,} series; eliminamos {len(original)-n_total} filas.')
print(f'2. Protegimos {ids_cruzados} IDs compartidos usando la clave compuesta; quedaron {catalogo.duplicated(CLAVE).sum()} duplicados.')
print(f'3. Obtuvimos popularidad mediana {catalogo["popularity"].median():.3f} y valoración media {catalogo["valoracion"].mean():.3f}/10.')
print(f'4. Contamos {paises_validos["país"].nunique()} países y {catalogo.loc[idioma_real, "language"].nunique()} idiomas reales; lideró {lider["language"]} con {int(lider["cantidad"]):,} títulos.')
print(f'5. Calculamos retorno para {int(ambos.sum()):,} películas ({100*ambos.sum()/n_peliculas:.2f}%); mediana {peliculas_finanzas["retorno"].median():.3f}x y {int(sospechoso.sum())} presupuestos marcados.')

1. Conservamos 31,991 títulos: 16,000 películas y 15,991 series; eliminamos 0 filas.
2. Protegimos 397 IDs compartidos usando la clave compuesta; quedaron 0 duplicados.
3. Obtuvimos popularidad mediana 22.071 y valoración media 6.630/10.
4. Contamos 147 países y 82 idiomas reales; lideró en con 13,969 títulos.
5. Calculamos retorno para 3,540 películas (22.12%); mediana 1.700x y 55 presupuestos marcados.
1. Conservamos 31,991 títulos: 16,000 películas y 15,991 series; eliminamos 0 filas.
2. Protegimos 397 IDs compartidos usando la clave compuesta; quedaron 0 duplicados.
3. Obtuvimos popularidad mediana 22.071 y valoración media 6.630/10.
4. Contamos 147 países y 82 idiomas reales; lideró en con 13,969 títulos.
5. Calculamos retorno para 3,540 películas (22.12%); mediana 1.700x y 55 presupuestos marcados.


In [ ]:


























































































# CELDA DE REVISIÓN: resume todo en una sola salida
def titulo(t):
    print('\n' + '=' * 60); print(t); print('=' * 60)

titulo('1. FILAS Y COLUMNAS')
print('original:', original.shape, '| catalogo:', catalogo.shape)
print('tipos:', conteo_tipo.to_dict())
print('IDs compartidos:', ids_cruzados, '| duplicados clave:', int(catalogo.duplicated(CLAVE).sum()))
print('columnas catalogo:', list(catalogo.columns))

titulo('2. VALORACIÓN')
print('vote_average == 0:', int((catalogo['vote_average'] == 0).sum()))
print('vote_count == 0:', int((catalogo['vote_count'] == 0).sum()))
print('valoracion vacía:', int(catalogo['valoracion'].isna().sum()))
print('valoracion con datos:', int(catalogo['valoracion'].notna().sum()))
print('valoracion media:', round(catalogo['valoracion'].mean(), 3))
print('valoracion en generos.csv:', 'valoracion' in generos.columns,
      '| en paises.csv:', 'valoracion' in paises.columns)

titulo('3. KPIs')
print(kpis.to_string(index=False))

titulo('4. FINANZAS (solo películas)')
print(cobertura.to_string(index=False))
print('presupuestos sospechosos:', int(sospechoso.sum()))
print('series con finanzas (debe ser 0):',
      int(catalogo.loc[~es_pelicula, ['budget', 'revenue', 'retorno']].notna().sum().sum()))
print('retorno: min', round(catalogo['retorno'].min(), 4),
      '| mediana', round(catalogo['retorno'].median(), 3),
      '| max', round(catalogo['retorno'].max(), 1))

titulo('5. TOP 10 (popularidad, vote_count >= 100)')
print(top10[['title', 'type', 'popularity', 'vote_count', 'valoracion' if 'valoracion' in top10 else 'vote_average', 'release_year']].to_string(index=False)
      if 'valoracion' in top10 else top10[['title', 'type', 'popularity', 'vote_count', 'vote_average', 'release_year']].to_string(index=False))

titulo('6. SERIES VS PELÍCULAS')
print(tabla_tipos.to_string())

titulo('7. GÉNEROS (p75 popularidad = %.2f | p35 cantidad = %.1f)' % (p75, p35))
print(tabla_generos[['género', 'cantidad', 'mediana_popularidad', 'nicho_destacado']].to_string(index=False))
print('Nichos:', tabla_generos.loc[tabla_generos['nicho_destacado'], 'género'].tolist())

titulo('8. IDIOMAS TOP 12')
print(tabla_idiomas.to_string(index=False))

titulo('9. PAÍSES TOP 12')
print(tabla_paises.to_string(index=False))

titulo('10. ARCHIVOS EXPORTADOS')
for nombre, tabla in salidas.items():
    print(nombre, tabla.shape)
print('generos.csv columnas:', list(generos.columns))
print('paises.csv columnas:', list(paises.columns))

titulo('11. NULOS EN COLUMNAS CLAVE')
print(catalogo[['show_id', 'type', 'title', 'country', 'genres', 'language', 'release_year',
                'popularity', 'vote_count', 'vote_average']].isna().sum().to_string())
print('release_year min/max:', int(catalogo['release_year'].min()), int(catalogo['release_year'].max()))


1. FILAS Y COLUMNAS
original: (31991, 16) | catalogo: (31991, 19)
tipos: {'Movie': 16000, 'TV Show': 15991}
IDs compartidos: 397 | duplicados clave: 0
columnas catalogo: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue', 'valoracion', 'retorno', 'budget_sospechoso']

2. VALORACIÓN
vote_average == 0: 4564
vote_count == 0: 4560
valoracion vacía: 4560
valoracion con datos: 27431
valoracion media: 6.63
valoracion en generos.csv: True | en paises.csv: True

3. KPIs
                           indicador       valor
                       Total títulos       31991
                       Películas (%)   50.014066
                          Series (%)   49.985934
                 Mediana popularidad      22.071
Valoración media (títulos con votos)    6.630076
                   Títulos con votos       27431
             Países reales distintos         147
 